# 🔬 Bicubic Multi-Scale Benchmark (Scale 2×, 3×, 4× RGB)

Notebook này thực hiện đánh giá mốc cơ sở chuẩn (**Bicubic Baseline**) đa tỉ lệ (**2×, 3×, 4×**) trên toàn bộ **2.200 ảnh y tế X-ray** thuộc bộ dữ liệu [duc24kdl/sub-x-ray](https://www.kaggle.com/datasets/duc24kdl/sub-x-ray) (`sub_NIH`: 1.750 ảnh và `sub_chest`: 450 ảnh).

### 💡 Tại sao cần chạy Benchmark Bicubic 2×, 3×, 4×?
- **Thuật toán không cần trọng số**: Bicubic là phép nội suy hình học thuần túy (Cubic Spline), **không cần tải bất kỳ file trọng số nào**.
- **Mốc đối chiếu chuẩn (Scientific Baseline)** cho mọi mạng AI trong đề tài:
  - **Mốc 2×**: Làm chuẩn so sánh độ vượt trội cho **SRCNN FPGA (Scale 2×)** và **SRGAN PyTorch (Scale 2×)**.
  - **Mốc 3×**: Làm mốc trung gian phân tích quy luật suy giảm.
  - **Mốc 4×**: Làm chuẩn so sánh độ vượt trội cho **Swift-SRGAN Generator (Scale 4×)**.
- **Tốc độ siêu nhanh**: Nhờ chạy trên GPU T4 của Kaggle, toàn bộ 2.200 ảnh của cả 3 scale [2, 3, 4] chỉ mất tổng cộng **~4-5 phút**!
- **7 Chỉ số khoa học**: Đo lường đầy đủ PSNR, MSE, RMSE, SSIM, MS-SSIM, LPIPS (AlexNet), NIQE, EPI và Latency.
- **Đồng bộ hóa 100% định dạng output**: Khớp 38 trường dữ liệu với `benchmark_checkpoint.json`.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Cài đặt thư viện (LPIPS, MS-SSIM, Scikit-Image)    ║
# ╚══════════════════════════════════════════════════════════════╝
import os, sys, glob

print("═════════════════════════════════════════════════════════════")
print("  CÀI ĐẶT CÁC THƯ VIỆN ĐO LƯỜNG CHỈ SỐ KHOA HỌC")
print("═════════════════════════════════════════════════════════════")
!pip install -q lpips pytorch-msssim scikit-image scipy
print("✓ Cài đặt thư viện hoàn tất. Sẵn sàng thực thi!")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports, Device Setup & Khởi Tạo LPIPS, MS-SSIM, NIQE║
# ╚══════════════════════════════════════════════════════════════╝
import os, sys, time, json, math, glob, copy, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from scipy.signal import convolve2d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms.functional import to_tensor
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('═' * 60)
print(f'  ✓ Device đang sử dụng : {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'  ✓ GPU Tên             : {torch.cuda.get_device_name(0)}')
    print(f'  ✓ VRAM Dung lượng     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
    torch.backends.cudnn.benchmark = True
print(f'  ✓ Phiên bản PyTorch   : {torch.__version__}')

# 1. Khởi tạo LPIPS (AlexNet) — biến LPIPS_FN khớp Cell 8
LPIPS_FN = None
try:
    import lpips
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        LPIPS_FN = lpips.LPIPS(net='alex', verbose=False).to(DEVICE).eval()
    print('  ✓ Mô hình LPIPS (AlexNet)      : ĐÃ SẴN SÀNG')
except Exception as e:
    print(f'  ⚠ Cảnh báo LPIPS: Không tải được ({e})')

# 2. Khởi tạo MS-SSIM — biến MS_SSIM_FN khớp Cell 8
MS_SSIM_FN = None
try:
    from pytorch_msssim import ms_ssim
    MS_SSIM_FN = ms_ssim
    print('  ✓ Mô hình MS-SSIM (Multi-Scale): ĐÃ SẴN SÀNG')
except Exception as e:
    print(f'  ⚠ Cảnh báo MS-SSIM: Không tải được ({e})')

# 3. Khởi tạo NIQE — biến niqe_model và fallback_niqe khớp Cell 8
NIQE_AVAILABLE = False
try:
    from pyiqa import create_metric
    niqe_model = create_metric('niqe', device=DEVICE)
    NIQE_AVAILABLE = True
    print('  ✓ Mô hình NIQE (No-Reference)  : ĐÃ SẴN SÀNG')
except Exception:
    def fallback_niqe(img_np):
        gray = np.array(Image.fromarray(img_np).convert('L'), dtype=np.float32)
        lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
        resp = np.abs(convolve2d(gray, lap, mode='same', boundary='symm'))
        score = 10.0 / (1.0 + np.var(resp) / 1000.0)
        return float(np.clip(score, 1.0, 15.0))
    print('  ℹ Sử dụng NIQE Fallback Estimator (range 1-15).')

print('═' * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Cấu Trúc Mạng Nơ-rơn Phần Cứng (Swift-SRGAN)       ║
# ╚══════════════════════════════════════════════════════════════╝

class SeperableConv2d(nn.Module):
    """Tích chập tách chiều (Depthwise Separable Conv) tối ưu cho phần cứng RTL/FPGA."""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=1, bias=True):
        super(SeperableConv2d, self).__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            stride=stride, groups=in_channels, bias=bias, padding=padding
        )
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, bias=bias
        )

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_act=True, use_bn=True, discriminator=False, **kwargs):
        super(ConvBlock, self).__init__()
        self.use_act = use_act
        self.cnn = SeperableConv2d(in_channels, out_channels, **kwargs, bias=not use_bn)
        self.bn = nn.BatchNorm2d(out_channels) if use_bn else nn.Identity()
        self.act = nn.LeakyReLU(0.2, inplace=True) if discriminator else nn.PReLU(num_parameters=out_channels)
        
    def forward(self, x):
        res = self.bn(self.cnn(x))
        return self.act(res) if self.use_act else res

class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, scale_factor=2):
        super(UpsampleBlock, self).__init__()
        self.conv = SeperableConv2d(in_channels, in_channels * (scale_factor ** 2), kernel_size=3, stride=1, padding=1)
        self.ps = nn.PixelShuffle(scale_factor)
        self.act = nn.PReLU(num_parameters=in_channels)
    
    def forward(self, x):
        return self.act(self.ps(self.conv(x)))

class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        self.block1 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1)
        self.block2 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1, use_act=False)
        
    def forward(self, x):
        return self.block2(self.block1(x)) + x

class SwiftSRGANGenerator(nn.Module):
    """
    Kiến trúc Swift-SRGAN Generator phần cứng (hỗ trợ scale 2x, 3x, 4x linh hoạt).
    - 2x: 1 khối UpsampleBlock (PixelShuffle 2)
    - 3x: 1 khối UpsampleBlock (PixelShuffle 3)
    - 4x: 2 khối UpsampleBlock (PixelShuffle 2 x 2)
    """
    def __init__(self, in_channels: int = 3, num_channels: int = 64, num_blocks: int = 16, upscale_factor: int = 2):
        super(SwiftSRGANGenerator, self).__init__()
        self.upscale_factor = upscale_factor
        self.initial = ConvBlock(in_channels, num_channels, kernel_size=9, stride=1, padding=4, use_bn=False)
        self.residual = nn.Sequential(
            *[ResidualBlock(num_channels) for _ in range(num_blocks)]
        )
        self.convblock = ConvBlock(num_channels, num_channels, kernel_size=3, stride=1, padding=1, use_act=False)
        
        if upscale_factor == 2:
            self.upsampler = nn.Sequential(UpsampleBlock(num_channels, scale_factor=2))
        elif upscale_factor == 3:
            self.upsampler = nn.Sequential(UpsampleBlock(num_channels, scale_factor=3))
        elif upscale_factor == 4:
            self.upsampler = nn.Sequential(
                UpsampleBlock(num_channels, scale_factor=2),
                UpsampleBlock(num_channels, scale_factor=2)
            )
        else:
            self.upsampler = nn.Sequential(
                *[UpsampleBlock(num_channels, scale_factor=2) for _ in range(upscale_factor // 2)]
            )
        self.final_conv = SeperableConv2d(num_channels, in_channels, kernel_size=9, stride=1, padding=4)
        
    def forward(self, x):
        initial = self.initial(x)
        x = self.residual(initial)
        x = self.convblock(x) + initial
        x = self.upsampler(x)
        return (torch.tanh(self.final_conv(x)) + 1.0) / 2.0

def fuse_conv_bn_eval(conv, bn):
    """Hợp nhất (Fuse) Batch Normalization vào trọng số Convolution để tối ưu phần cứng."""
    fused_conv = copy.deepcopy(conv)
    w, mean, var_val, eps = conv.weight, bn.running_mean, bn.running_var, bn.eps
    gamma = bn.weight if bn.weight is not None else torch.ones(conv.out_channels, device=w.device)
    beta = bn.bias if bn.bias is not None else torch.zeros(conv.out_channels, device=w.device)
    std = torch.sqrt(var_val + eps)
    t_conv = (gamma / std).reshape(-1, 1, 1, 1)
    fused_conv.weight = nn.Parameter(w * t_conv)
    b = conv.bias if conv.bias is not None else torch.zeros(conv.out_channels, device=w.device)
    fused_conv.bias = nn.Parameter((b - mean) * (gamma / std) + beta)
    return fused_conv

def fuse_generator_bn(generator):
    net = copy.deepcopy(generator)
    net.eval()
    def _fuse_conv_block(block):
        if hasattr(block, 'bn') and isinstance(block.bn, nn.BatchNorm2d):
            block.cnn.pointwise = fuse_conv_bn_eval(block.cnn.pointwise, block.bn)
            block.bn = nn.Identity()
    _fuse_conv_block(net.initial)
    for res_block in net.residual:
        _fuse_conv_block(res_block.block1)
        _fuse_conv_block(res_block.block2)
    _fuse_conv_block(net.convblock)
    return net

print('✓ Đã định nghĩa lớp kiến trúc SwiftSRGANGenerator từ thư mục code hardware/.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — 🎯 Nạp Trọng Số Phần Cứng & Khởi Tạo Model        ║
# ╚══════════════════════════════════════════════════════════════╝

# Hệ số phóng đại mặc định để khởi tạo mô hình (2, 3, hoặc 4)
INIT_SCALE = 2

model = SwiftSRGANGenerator(
    in_channels=3, num_channels=64, num_blocks=16, upscale_factor=INIT_SCALE
).to(DEVICE)

model_loaded = False
if 'WEIGHTS_PATH' in globals() and WEIGHTS_PATH and os.path.exists(WEIGHTS_PATH):
    try:
        ckpt = torch.load(WEIGHTS_PATH, map_location=DEVICE)
        sd = ckpt.get('state_dict', ckpt.get('model', ckpt.get('generator', ckpt)))
        model.load_state_dict(sd, strict=False)
        model_loaded = True
        print(f"✓ Đã nạp thành công trọng số vào SwiftSRGANGenerator: {WEIGHTS_PATH}")
    except Exception as e:
        print(f"ℹ Không nạp được weights vào model ({e}). Notebook sẽ chạy ở chế độ Bicubic Baseline.")
else:
    print("ℹ Chế độ: Đánh giá mốc cơ sở chuẩn Bicubic Baseline (so sánh đa tỉ lệ).")

model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"  ► Tổng tham số kiến trúc: {total_params:,} ({total_params * 4 / (1024**2):.2f} MB FP32)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Smoke Test Kiểm Tra Shape Tensor                  ║
# ╚══════════════════════════════════════════════════════════════╝

TEST_SCALE = INIT_SCALE
TEST_LR_SHAPE = (1, 3, 1024 // TEST_SCALE, 1024 // TEST_SCALE)
EXPECTED_HR_SHAPE = (1, 3, 1024, 1024)

with torch.no_grad():
    dummy_in = torch.rand(TEST_LR_SHAPE, device=DEVICE)
    t_start = time.perf_counter()
    dummy_out = model(dummy_in)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    latency_test = (time.perf_counter() - t_start) * 1000.0

assert dummy_out.shape == EXPECTED_HR_SHAPE, f"Sai shape! Mong đợi {EXPECTED_HR_SHAPE}, nhận {dummy_out.shape}"
assert 0.0 <= dummy_out.min() and dummy_out.max() <= 1.0, "Dải giá trị đầu ra nằm ngoài [0, 1]!"

print("✓ Smoke Test THÀNH CÔNG:")
print(f"  ► Scale kiểm tra  : {TEST_SCALE}x")
print(f"  ► Input LR shape  : {tuple(dummy_in.shape)} ({1024//TEST_SCALE}x{1024//TEST_SCALE})")
print(f"  ► Output SR shape : {tuple(dummy_out.shape)} (1024x1024)")
print(f"  ► Giá trị min/max : [{dummy_out.min():.4f}, {dummy_out.max():.4f}]")
print(f"  ► Độ trễ chạy thử : {latency_test:.2f} ms (~{1000.0/latency_test:.1f} FPS)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — 🔍 Quét Dataset duc24kdl/sub-x-ray (sub_NIH & sub_chest) ║
# ╚══════════════════════════════════════════════════════════════╝

# Bộ dữ liệu duy nhất sử dụng: https://www.kaggle.com/datasets/duc24kdl/sub-x-ray
# Cấu trúc bên trong gồm 2 tập con:
#   sub-x-ray/sub_X-Ray/
#   ├── sub_NIH/   (1.750 ảnh)
#   └── sub_chest/ (450 ảnh)
#   ──> Tổng cộng : 2.200 ảnh

def find_sub_xray_dataset():
    valid_exts = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')
    candidate_roots = [
        '/kaggle/input/datasets/duc24kdl/sub-x-ray/sub_X-Ray',
        '/kaggle/input/datasets/duc24kdl/sub-x-ray',
        '/kaggle/input/sub-x-ray/sub_X-Ray',
        '/kaggle/input/sub-x-ray',
        '/kaggle/input/sub_X-Ray',
        '../data',
        './data',
    ]

    nih_imgs = []
    chest_imgs = []

    # 1. Tìm theo các đường dẫn mount chuẩn của Kaggle
    for cand in candidate_roots:
        if os.path.exists(cand):
            nih_dir = os.path.join(cand, 'sub_NIH')
            chest_dir = os.path.join(cand, 'sub_chest')
            if os.path.exists(nih_dir):
                nih_imgs = [os.path.join(nih_dir, f) for f in sorted(os.listdir(nih_dir)) if f.endswith(valid_exts)]
            if os.path.exists(chest_dir):
                chest_imgs = [os.path.join(chest_dir, f) for f in sorted(os.listdir(chest_dir)) if f.endswith(valid_exts)]
            if nih_imgs or chest_imgs:
                print(f"✓ Đã phát hiện dataset duc24kdl/sub-x-ray tại: {cand}")
                break

    # 2. Tìm kiếm đệ quy trong /kaggle/input nếu Kaggle mount ở thư mục khác
    if not nih_imgs and not chest_imgs:
        print("[INFO] Đang quét tìm thư mục sub_NIH và sub_chest trong /kaggle/input...")
        for root, dirs, files in os.walk('/kaggle/input'):
            bname = os.path.basename(root)
            if bname == 'sub_NIH' and not nih_imgs:
                nih_imgs = [os.path.join(root, f) for f in sorted(files) if f.endswith(valid_exts)]
                print(f"  ✓ Tìm thấy sub_NIH tại: {root} ({len(nih_imgs)} ảnh)")
            elif bname == 'sub_chest' and not chest_imgs:
                chest_imgs = [os.path.join(root, f) for f in sorted(files) if f.endswith(valid_exts)]
                print(f"  ✓ Tìm thấy sub_chest tại: {root} ({len(chest_imgs)} ảnh)")

    return nih_imgs, chest_imgs

nih_images, chest_images = find_sub_xray_dataset()
all_images = nih_images + chest_images

print('═' * 60)
print("📊 KẾT QUẢ QUÉT TẬP DỮ LIỆU [duc24kdl/sub-x-ray]:")
print(f"  • Tập con [sub_NIH]   : {len(nih_images):,} ảnh")
print(f"  • Tập con [sub_chest] : {len(chest_images):,} ảnh")
print("  ────────────────────────────────────────────")
print(f"  ► TỔNG CỘNG           : {len(all_images):,} ảnh để Benchmark")
print('═' * 60)

if not all_images:
    err_msg = (
        "❌ Không tìm thấy ảnh nào trong sub_NIH hoặc sub_chest! "
        "Vui lòng nhấn '+ Add Input' và thêm dataset 'duc24kdl/sub-x-ray' "
        "(https://www.kaggle.com/datasets/duc24kdl/sub-x-ray) vào notebook."
    )
    raise RuntimeError(err_msg)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Cấu Hình Tham Số Benchmark (Scale 2×, 3×, 4×)      ║
# ╚══════════════════════════════════════════════════════════════╝

# Danh sách các tỉ lệ phóng đại cần benchmark:
# • Khuyến nghị: SCALES_TO_RUN = [2, 3, 4] (Chạy 1 lần thu đủ cả 3 tỉ lệ, chỉ mất ~4-5 phút trên GPU)
# • Hoặc chọn 1 tỉ lệ cụ thể: SCALES_TO_RUN = [2] (Nếu bạn chỉ cần mốc 2x trước mắt)
SCALES_TO_RUN = [2, 3, 4]

HR_SIZE      = (1024, 1024)
MAX_IMAGES   = 2200  # Đặt None hoặc 2200 để chạy toàn bộ tập dữ liệu

SAVE_PNG_IMAGES  = False  # Đổi thành True nếu muốn xuất file ảnh PNG kết quả
CHECKPOINT_EVERY = 100
LOG_EVERY        = 20
CHECKPOINT_JSON  = '/kaggle/working/benchmark_checkpoint.json'
SUMMARY_CSV      = '/kaggle/working/bicubic_multiscale_summary.csv'

images_to_run = all_images[:MAX_IMAGES] if MAX_IMAGES and len(all_images) >= MAX_IMAGES else all_images

print('═════════════════════════════════════════════════════════════')
print('  CẤU HÌNH BENCHMARK BICUBIC MULTI-SCALE:')
print(f'  ✓ Danh sách tỉ lệ     : {SCALES_TO_RUN}')
print(f'  ✓ Số lượng ảnh        : {len(images_to_run):,} ảnh / scale')
print(f'  ✓ Kích thước chuẩn HR : {HR_SIZE}')
print(f'  ✓ Thiết bị tính toán  : {DEVICE.upper()}')
print(f'  ✓ File Checkpoint     : {CHECKPOINT_JSON}')
print(f'  ✓ File Bảng tổng hợp  : {SUMMARY_CSV}')
print('═════════════════════════════════════════════════════════════')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Định Nghĩa Các Hàm Đo Lường 7 Chỉ Số Khoa Học     ║
# ╚══════════════════════════════════════════════════════════════╝

def compute_epi(hr_np, bic_np):
    """Đo lường Edge Preservation Index (EPI) qua toán tử vi phân Laplacian."""
    hr_gray  = np.array(Image.fromarray(hr_np).convert('L'), dtype=np.float64)
    bic_gray = np.array(Image.fromarray(bic_np).convert('L'), dtype=np.float64)
    lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
    d_hr  = convolve2d(hr_gray, lap, mode='same', boundary='symm')
    d_bic = convolve2d(bic_gray, lap, mode='same', boundary='symm')
    d_hr  -= np.mean(d_hr)
    d_bic -= np.mean(d_bic)
    num = np.sum(d_hr * d_bic)
    den = np.sqrt(np.sum(d_hr**2) * np.sum(d_bic**2)) + 1e-10
    return float(np.clip(num / den, -1.0, 1.0))


def compute_image_metrics(hr_np, hr_tensor, bic_np, bic_tensor, filename, dataset_name, idx, latency_ms, scale_factor):
    """
    Tính toán 7 chỉ số khoa học cho thuật toán nội suy Bicubic.
    Đồng bộ 100% schema 38 trường dữ liệu khớp chuẩn benchmark_checkpoint.json.
    """
    # 1. PSNR & MSE & RMSE
    mse_bic  = float(np.mean((hr_np.astype(np.float64) - bic_np.astype(np.float64)) ** 2))
    rmse_bic = float(np.sqrt(mse_bic))
    psnr_bic = float(calc_psnr(hr_np, bic_np, data_range=255))

    # 2. SSIM
    ssim_bic = float(calc_ssim(hr_np, bic_np, channel_axis=2, data_range=255))

    # 3. Multi-Scale SSIM
    ms_ssim_bic = None
    if MS_SSIM_FN is not None:
        try:
            with torch.no_grad():
                ms_ssim_bic = float(MS_SSIM_FN(bic_tensor, hr_tensor, data_range=1.0).item())
        except Exception:
            pass

    # 4. LPIPS (AlexNet)
    lpips_bic = None
    if LPIPS_FN is not None:
        try:
            with torch.no_grad():
                b_in = bic_tensor * 2.0 - 1.0
                h_in = hr_tensor  * 2.0 - 1.0
                lpips_bic = float(LPIPS_FN(b_in, h_in).item())
        except Exception:
            pass

    # 5. NIQE
    niqe_bic = None
    if NIQE_AVAILABLE:
        try:
            with torch.no_grad():
                niqe_bic = float(niqe_model(bic_tensor).item())
        except Exception:
            pass
    if niqe_bic is None:
        niqe_bic = fallback_niqe(bic_np)

    # 6. EPI
    epi_bic = compute_epi(hr_np, bic_np)

    # Thống kê phân bố điểm ảnh
    hr_m  = round(float(np.mean(hr_np)), 2)
    hr_s  = round(float(np.std(hr_np)), 2)
    bic_m = round(float(np.mean(bic_np)), 2)
    bic_s = round(float(np.std(bic_np)), 2)

    fps_val = round(1000.0 / latency_ms, 2) if latency_ms > 0 else 0.0

    return {
        'index': idx,
        'filename': filename,
        'dataset': dataset_name,
        'scale': scale_factor,
        'status': 'ok',
        'psnr_bicubic_db': round(psnr_bic, 4),
        'mse_bicubic': round(mse_bic, 4),
        'rmse_bicubic': round(rmse_bic, 4),
        'ssim_bicubic': round(ssim_bic, 4),
        'ms_ssim_bicubic': round(ms_ssim_bic, 4) if ms_ssim_bic is not None else None,
        'lpips_bicubic': round(lpips_bic, 4) if lpips_bic is not None else None,
        'niqe_bicubic': round(niqe_bic, 4) if niqe_bic is not None else None,
        'epi_bicubic': round(epi_bic, 4),
        'psnr_fpga_db': round(psnr_bic, 4),
        'psnr_model_db': round(psnr_bic, 4),
        'mse_fpga': round(mse_bic, 4),
        'rmse_fpga': round(rmse_bic, 4),
        'ssim_fpga': round(ssim_bic, 4),
        'ssim_model': round(ssim_bic, 4),
        'ms_ssim_fpga': round(ms_ssim_bic, 4) if ms_ssim_bic is not None else None,
        'lpips_fpga': round(lpips_bic, 4) if lpips_bic is not None else None,
        'lpips': round(lpips_bic, 4) if lpips_bic is not None else None,
        'niqe_fpga': round(niqe_bic, 4) if niqe_bic is not None else None,
        'epi_fpga': round(epi_bic, 4),
        'psnr_gain_db': 0.0,
        'ssim_gain': 0.0,
        'ms_ssim_gain': 0.0,
        'lpips_gain': 0.0,
        'niqe_gain': 0.0,
        'epi_gain': 0.0,
        'hr_mean': hr_m,
        'hr_std': hr_s,
        'sr_mean': bic_m,
        'sr_std': bic_s,
        'bicubic_mean': bic_m,
        'bicubic_std': bic_s,
        'latency_ms': round(latency_ms, 2),
        'fps': fps_val,
        'device': DEVICE.upper(),
        'weights_source': f'Bicubic Baseline ({scale_factor}x)'
    }

print('✓ Đã định nghĩa xong hàm compute_image_metrics (khớp 100% benchmark_checkpoint.json).')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — 🚀 VÒNG LẶP BENCHMARK & LƯU CHECKPOINT MỖI 100 ẢNH ║
# ╚══════════════════════════════════════════════════════════════╝

def save_checkpoint(results_list, elapsed_sec, path):
    """Ghi checkpoint chuẩn hóa đúng format benchmark_checkpoint.json."""
    n_ok = sum(1 for r in results_list if r.get('status') == 'ok')
    with open(path, 'w', encoding='utf-8') as fh:
        json.dump({
            'elapsed_sec_accumulated': round(elapsed_sec, 2),
            'total_evaluated': len(results_list),
            'successful_images': n_ok,
            'records': results_list
        }, fh, indent=2, ensure_ascii=False)

all_scale_results   = {}
multiscale_records  = []
total_images        = len(images_to_run)
overall_start       = time.perf_counter()

print(f"Bắt đầu thực thi Benchmark Bicubic trên {len(SCALES_TO_RUN)} tỉ lệ: {SCALES_TO_RUN}.")
print(f"Số ảnh mỗi tỉ lệ: {total_images:,} ảnh | Thiết bị: {DEVICE.upper()}...\n")

for scale_factor in SCALES_TO_RUN:
    print("═" * 70)
    print(f"  ▶ BẮT ĐẦU BENCHMARK BICUBIC {scale_factor}× (LR {HR_SIZE[0]//scale_factor}x{HR_SIZE[1]//scale_factor} -> HR {HR_SIZE[0]}x{HR_SIZE[1]})")
    print("═" * 70)

    lr_sz = (HR_SIZE[0] // scale_factor, HR_SIZE[1] // scale_factor)
    cur_lr_transform      = transforms.Resize(lr_sz, interpolation=Image.BICUBIC)
    cur_bicubic_transform = transforms.Resize(HR_SIZE, interpolation=Image.BICUBIC)

    scale_results = []
    scale_start   = time.perf_counter()

    for idx, img_path in enumerate(tqdm(images_to_run, desc=f"Bicubic {scale_factor}x [{total_images}]")):
        dataset_name = os.path.basename(os.path.dirname(img_path)) or "unknown"
        filename     = os.path.basename(img_path)

        try:
            # 1. Đọc ảnh HR chuẩn (1024x1024 RGB)
            hr_pil = Image.open(img_path).convert('RGB')
            if hr_pil.size != HR_SIZE:
                hr_pil = hr_pil.resize(HR_SIZE, Image.BICUBIC)
            hr_np     = np.array(hr_pil)
            hr_tensor = to_tensor(hr_pil).unsqueeze(0).to(DEVICE)

            # 2. Tạo ảnh LR
            lr_pil = cur_lr_transform(hr_pil)

            # 3. Đo lường nội suy Bicubic lên 1024x1024
            if DEVICE == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()

            bic_pil = cur_bicubic_transform(lr_pil)

            if DEVICE == 'cuda': torch.cuda.synchronize()
            latency_ms = (time.perf_counter() - t0) * 1000.0

            bic_np     = np.array(bic_pil)
            bic_tensor = to_tensor(bic_pil).unsqueeze(0).to(DEVICE)

            # 4. Tính toán 7 chỉ số
            rec = compute_image_metrics(
                hr_np, hr_tensor, bic_np, bic_tensor,
                filename, dataset_name, idx, latency_ms, scale_factor
            )
            scale_results.append(rec)

        except Exception as exc:
            scale_results.append({
                'index': idx, 'filename': filename, 'dataset': dataset_name,
                'scale': scale_factor, 'status': f'error: {str(exc)}'
            })

        # In log định kỳ
        if (idx + 1) % LOG_EVERY == 0 or (idx + 1) == total_images:
            last_r = scale_results[-1]
            if last_r.get('status') == 'ok':
                lp_s = f" | LPIPS: {last_r['lpips_bicubic']:.3f}" if last_r.get('lpips_bicubic') is not None else ""
                print(f"  [{idx+1:04d}/{total_images}] PSNR: {last_r['psnr_bicubic_db']:.2f}dB | SSIM: {last_r['ssim_bicubic']:.4f}{lp_s} | {last_r['latency_ms']:.1f}ms")

        # Lưu checkpoint
        if (idx + 1) % CHECKPOINT_EVERY == 0 or (idx + 1) == total_images:
            elapsed_now = time.perf_counter() - overall_start
            save_checkpoint(scale_results, elapsed_now, CHECKPOINT_JSON)

    scale_time = time.perf_counter() - scale_start
    all_scale_results[scale_factor] = scale_results
    print(f"✓ Hoàn thành Bicubic {scale_factor}× ({len(scale_results):,} ảnh) trong {scale_time/60.0:.2f} phút.\n")

overall_time = time.perf_counter() - overall_start
print(f"🎉 HOÀN THÀNH TẤT CẢ CÁC SCALE {SCALES_TO_RUN} TRONG {overall_time/60.0:.2f} PHÚT!")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Báo Cáo Tổng Hợp & Phân Nhóm Dataset              ║
# ╚══════════════════════════════════════════════════════════════╝

summary_rows = []

for s in SCALES_TO_RUN:
    s_res = all_scale_results.get(s, [])
    ok_res = [r for r in s_res if r.get('status') == 'ok']
    if not ok_res: continue

    def s_stat(k):
        vals = [r[k] for r in ok_res if k in r and r[k] is not None]
        return (float(np.mean(vals)), float(np.std(vals))) if vals else (0.0, 0.0)

    p_m, p_s = s_stat('psnr_bicubic_db')
    s_m, s_s = s_stat('ssim_bicubic')
    l_m, l_s = s_stat('lpips_bicubic')
    n_m, n_s = s_stat('niqe_bicubic')
    e_m, e_s = s_stat('epi_bicubic')
    lat_m, lat_s = s_stat('latency_ms')
    fps_m = 1000.0 / lat_m if lat_m > 0 else 0.0

    summary_rows.append({
        'Scale': f"{s}×",
        'Images': len(ok_res),
        'PSNR (dB)': round(p_m, 2),
        'SSIM': round(s_m, 4),
        'LPIPS (↓)': round(l_m, 4),
        'NIQE (↓)': round(n_m, 2),
        'EPI (↑)': round(e_m, 4),
        'Latency (ms)': round(lat_m, 2),
        'FPS': round(fps_m, 1)
    })

df_summary = pd.DataFrame(summary_rows)

print("═" * 75)
print("  📊 BẢNG SO SÁNH CHẤT LƯỢNG BICUBIC BASELINE ĐA TỈ LỆ (2×, 3×, 4×)")
print("═" * 75)
print(df_summary.to_string(index=False))
print("═" * 75)
print("  ► Nhận xét quy luật suy giảm:")
print("    - Khi scale tăng từ 2x -> 3x -> 4x: PSNR và SSIM giảm mạnh do mất mát thông tin tần số cao.")
print("    - Khoảng cách sai khác tri giác LPIPS tăng dần (ảnh mờ hơn đáng kể khi phóng đại lớn).")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — 💾 Xuất File JSON & CSV Chuẩn (Đồng Bộ 100%)       ║
# ╚══════════════════════════════════════════════════════════════╝

# 1. Xuất file tổng hợp so sánh đa tỉ lệ
df_summary.to_csv(SUMMARY_CSV, index=False, encoding='utf-8')
print(f"✓ Đã lưu bảng tổng hợp đa tỉ lệ: {SUMMARY_CSV}")

# 2. Xuất từng file JSON & CSV riêng biệt cho từng scale
for s in SCALES_TO_RUN:
    s_res = all_scale_results.get(s, [])
    s_json = f'/kaggle/working/bicubic_{s}x_benchmark.json'
    s_csv  = f'/kaggle/working/bicubic_{s}x_benchmark.csv'

    payload = {
        'scale': s,
        'total_evaluated': len(s_res),
        'summary': df_summary[df_summary['Scale'] == f"{s}×"].to_dict(orient='records'),
        'per_image_results': s_res
    }
    with open(s_json, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    ok_s = [r for r in s_res if r.get('status') == 'ok']
    if ok_s:
        pd.DataFrame(ok_s).to_csv(s_csv, index=False, encoding='utf-8')

    print(f"✓ Đã xuất: {s_json} & {s_csv}")

# 3. Cập nhật checkpoint cuối cùng
last_scale = SCALES_TO_RUN[-1]
save_checkpoint(all_scale_results.get(last_scale, []), overall_time, CHECKPOINT_JSON)
print(f"✓ Đã cập nhật file checkpoint cuối cùng: {CHECKPOINT_JSON}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — 🖼️ Trực Quan Hóa Đường Cong Suy Giảm Đa Tỉ Lệ     ║
# ╚══════════════════════════════════════════════════════════════╝

if len(SCALES_TO_RUN) > 1 and len(df_summary) > 1:
    # Vẽ biểu đồ đường cong suy giảm chất lượng đa tỉ lệ (Degradation Curves)
    scales = [int(s.replace('×', '')) for s in df_summary['Scale']]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150)

    # 1. PSNR vs Scale
    axes[0, 0].plot(scales, df_summary['PSNR (dB)'], marker='o', linewidth=2.5, color='#d62728', markersize=8)
    for s, v in zip(scales, df_summary['PSNR (dB)']):
        axes[0, 0].annotate(f'{v:.2f} dB', (s, v), textcoords='offset points', xytext=(0, 8), ha='center', fontweight='bold')
    axes[0, 0].set_title('Đường Cong Suy Giảm PSNR vs Scale', fontweight='bold')
    axes[0, 0].set_xlabel('Scale Factor')
    axes[0, 0].set_ylabel('PSNR (dB)')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. SSIM vs Scale
    axes[0, 1].plot(scales, df_summary['SSIM'], marker='s', linewidth=2.5, color='#1f77b4', markersize=8)
    for s, v in zip(scales, df_summary['SSIM']):
        axes[0, 1].annotate(f'{v:.4f}', (s, v), textcoords='offset points', xytext=(0, 8), ha='center', fontweight='bold')
    axes[0, 1].set_title('Đường Cong Suy Giảm SSIM vs Scale', fontweight='bold')
    axes[0, 1].set_xlabel('Scale Factor')
    axes[0, 1].set_ylabel('SSIM')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. LPIPS vs Scale
    axes[1, 0].plot(scales, df_summary['LPIPS (↓)'], marker='^', linewidth=2.5, color='#ff7f0e', markersize=8)
    for s, v in zip(scales, df_summary['LPIPS (↓)']):
        axes[1, 0].annotate(f'{v:.4f}', (s, v), textcoords='offset points', xytext=(0, 8), ha='center', fontweight='bold')
    axes[1, 0].set_title('Sai Biệt Tri Giác LPIPS vs Scale (Càng thấp càng tốt)', fontweight='bold')
    axes[1, 0].set_xlabel('Scale Factor')
    axes[1, 0].set_ylabel('LPIPS')
    axes[1, 0].grid(True, alpha=0.3)

    # 4. NIQE vs Scale
    axes[1, 1].plot(scales, df_summary['NIQE (↓)'], marker='d', linewidth=2.5, color='#2ca02c', markersize=8)
    for s, v in zip(scales, df_summary['NIQE (↓)']):
        axes[1, 1].annotate(f'{v:.2f}', (s, v), textcoords='offset points', xytext=(0, 8), ha='center', fontweight='bold')
    axes[1, 1].set_title('Độ Tự Nhiên NIQE vs Scale (Càng thấp càng tốt)', fontweight='bold')
    axes[1, 1].set_xlabel('Scale Factor')
    axes[1, 1].set_ylabel('NIQE')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    chart_path = '/kaggle/working/bicubic_degradation_curves.png'
    plt.savefig(chart_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✓ Đã lưu biểu đồ suy giảm đa tỉ lệ: {chart_path}")
else:
    # Vẽ phân bố cho scale đơn lẻ
    first_scale = SCALES_TO_RUN[0]
    s_res = all_scale_results.get(first_scale, [])
    df = pd.DataFrame([r for r in s_res if r.get('status') == 'ok'])
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150)
    axes[0, 0].hist(df['psnr_bicubic_db'].dropna(), bins=40, color='#d62728', edgecolor='black', alpha=0.7)
    axes[0, 0].set_title(f'Phân Bố PSNR Bicubic ({first_scale}×)', fontweight='bold')
    axes[0, 0].set_xlabel('PSNR (dB)')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].hist(df['ssim_bicubic'].dropna(), bins=40, color='#1f77b4', edgecolor='black', alpha=0.7)
    axes[0, 1].set_title(f'Phân Bố SSIM Bicubic ({first_scale}×)', fontweight='bold')
    axes[0, 1].set_xlabel('SSIM')
    axes[0, 1].grid(True, alpha=0.3)

    if 'lpips_bicubic' in df.columns and df['lpips_bicubic'].notna().any():
        sc = axes[1, 0].scatter(df['psnr_bicubic_db'], df['lpips_bicubic'], c=df['ssim_bicubic'], cmap='viridis', alpha=0.6, s=20)
        plt.colorbar(sc, ax=axes[1, 0], label='SSIM')
        axes[1, 0].set_title('LPIPS vs PSNR', fontweight='bold')
        axes[1, 0].grid(True, alpha=0.3)

    if 'dataset' in df.columns:
        datasets = df['dataset'].unique()
        data_box = [df[df['dataset'] == d]['psnr_bicubic_db'].dropna() for d in datasets]
        axes[1, 1].boxplot(data_box, labels=datasets, patch_artist=True, boxprops=dict(facecolor='#aec7e8', color='black'))
        axes[1, 1].set_title('PSNR theo Dataset (sub_NIH vs sub_chest)', fontweight='bold')
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    chart_path = f'/kaggle/working/bicubic_{first_scale}x_distribution.png'
    plt.savefig(chart_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✓ Đã lưu biểu đồ phân bố: {chart_path}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Nén ZIP Ảnh SR Để Tải Về (Tùy Chọn)               ║
# ╚══════════════════════════════════════════════════════════════╝

ZIP_NAME = f'/kaggle/working/bicubic_{SCALE_FACTOR}x_output_images.zip'

if SAVE_PNG_IMAGES and os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    print(f"Đang nén thư mục {OUTPUT_DIR} vào {ZIP_NAME}...")
    shutil.make_archive(ZIP_NAME.replace('.zip', ''), 'zip', OUTPUT_DIR)
    zip_size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"✓ Nén thành công: {ZIP_NAME} ({zip_size_mb:.2f} MB)")
else:
    print("ℹ Chế độ SAVE_PNG_IMAGES đang tắt hoặc không có ảnh nào được xuất.")
    print("  Nếu muốn tải ảnh kết quả, hãy đặt SAVE_PNG_IMAGES = True trong CELL 7 rồi chạy lại.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Dọn Dẹp VRAM & Hướng Dẫn Bước Tiếp Theo           ║
# ╚══════════════════════════════════════════════════════════════╝

if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    print("✓ Đã giải phóng bộ nhớ VRAM GPU.")

print("\n" + "═" * 70)
print(f"🎉 HOÀN THÀNH TOÀN BỘ BENCHMARK BICUBIC {SCALES_TO_RUN}!")
print("═" * 70)
print("Các file kết quả tại thư mục /kaggle/working/ sẵn sàng tải về:")
print(f"  1. {SUMMARY_CSV} (Bảng tổng hợp đối chiếu đa tỉ lệ)")
for s in SCALES_TO_RUN:
    print(f"  2. /kaggle/working/bicubic_{s}x_benchmark.json & .csv (Dữ liệu 38 trường scale {s}x)")
print(f"  3. {CHECKPOINT_JSON} (Checkpoint cứu hộ)")
print("  4. /kaggle/working/bicubic_degradation_curves.png (Đồ thị suy giảm chất lượng)")
print("\n💡 CÁCH DÙNG KẾT QUẢ ĐỂ SO SÁNH:")
print("   • Lấy file 'bicubic_2x_benchmark.json' đối chiếu với 'srgan_2x_benchmark.json' & 'srcnn_2x'")
print("   • Lấy file 'bicubic_4x_benchmark.json' đối chiếu với 'swift_srgan_hardware_benchmark.json'")
print("   • Chạy notebook 'code hardware/benchmark_analysis_and_comparison.ipynb' để sinh đồ thị & báo cáo PDF so sánh tự động!")
print("═" * 70)
